# Lab 1 — Data Analysis and Visualization
### Dataset Inspection & Initial Data Profiling — UCI Adult Income Dataset
**Name:** Faseeh &nbsp;&nbsp; **Roll No:** 24i-6517 &nbsp;&nbsp; **Section:** DS-A

In [1]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

## Task 1 — Fix Hidden Missing Values


In [3]:
df = pd.read_csv('adult.csv')

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
# isnull().sum() looks clean — but the data has hidden missing values encoded as '?'
print('Missing values BEFORE fix:', df.isnull().sum().sum())

df.replace('?', np.nan, inplace=True)

print('Missing values AFTER fix:', df.isnull().sum().sum())

Missing values BEFORE fix: 0
Missing values AFTER fix: 4262


.

## Task 2 — Quantify Missingness


In [5]:
missing_count = df.isnull().sum().sort_values(ascending=False)
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({'missing_count': missing_count, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0]
missing_summary

,missing_count,missing_pct
occupation,1843,5.660146
workclass,1836,5.638647
native.country,583,1.790486


## Task 3 — Handle Missing Values


In [6]:
df['workclass'] = df['workclass'].fillna('Unknown')
df['occupation'] = df['occupation'].fillna('Unknown')
df['native.country'] = df['native.country'].fillna(df['native.country'].mode()[0])

print('Remaining missing values:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nTotal missing values left:', df.isnull().sum().sum())

Remaining missing values:
 Series([], dtype: int64)

Total missing values left: 0


## Task 4 — Detect Duplicates


In [7]:
exact_dupes = df.duplicated().sum()
print('Exact duplicate rows:', exact_dupes)

cols_no_label = [c for c in df.columns if c != 'income']
dupes_ignoring_label = df.duplicated(subset=cols_no_label).sum()
print('Duplicate rows ignoring the income label:', dupes_ignoring_label)

Exact duplicate rows: 24
Duplicate rows ignoring the income label: 25


In [8]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print('Shape after dropping exact duplicates:', df.shape)

Shape after dropping exact duplicates: (32537, 15)


## Task 5 — Fix Inconsistent Categorical Data


In [9]:
for col in ['education', 'marital.status', 'native.country']:
    print(f"--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].unique())
    print()

--- education (16 unique values) ---
['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'Preschool']

--- marital.status (7 unique values) ---
['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse'
 'Married-spouse-absent' 'Married-AF-spouse']

--- native.country (41 unique values) ---
['United-States' 'Mexico' 'Greece' 'Vietnam' 'China' 'Taiwan' 'India'
 'Philippines' 'Trinadad&Tobago' 'Canada' 'South' 'Holand-Netherlands'
 'Puerto-Rico' 'Poland' 'Iran' 'England' 'Germany' 'Italy' 'Japan' 'Hong'
 'Honduras' 'Cuba' 'Ireland' 'Cambodia' 'Peru' 'Nicaragua'
 'Dominican-Republic' 'Haiti' 'El-Salvador' 'Hungary' 'Columbia'
 'Guatemala' 'Jamaica' 'Ecuador' 'France' 'Yugoslavia' 'Scotland'
 'Portugal' 'Laos' 'Thailand' 'Outlying-US(Guam-USVI-etc)']



In [10]:
for col in ['education', 'marital.status', 'native.country']:
    df[col] = df[col].str.strip()

print('Stripped whitespace from education, marital.status, native.country.')

Stripped whitespace from education, marital.status, native.country.


## Task 6 — Descriptive Statistics

In [ ]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32537.000000,3.253700e+04,32537.000000,32537.000000,32537.000000,32537.000000
mean,38.585549,1.897808e+05,10.081815,1078.443741,87.368227,40.440329
std,13.637984,1.055565e+05,2.571633,7387.957424,403.101833,12.346889
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.369930e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [11]:
df.describe(include='object')

,workclass,education,marital.status,occupation,relationship,race,sex,native.country,income
count,32537,32537,32537,32537,32537,32537,32537,32537,32537
unique,9,16,7,15,6,5,2,41,2
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
freq,22673,10494,14970,4136,13187,27795,21775,29735,24698


In [12]:
age_mean, age_median = df['age'].mean(), df['age'].median()
hrs_mean, hrs_median = df['hours.per.week'].mean(), df['hours.per.week'].median()

print(f'Age       -> mean: {age_mean:.2f}, median: {age_median}')
print(f'Hours/week -> mean: {hrs_mean:.2f}, median: {hrs_median}')

Age       -> mean: 38.59, median: 37.0
Hours/week -> mean: 40.44, median: 40.0


## Task 7 — Frequency Analysis

In [13]:
print('Most common occupation:')
print(df['occupation'].value_counts().head(1))

Most common occupation:
occupation
Prof-specialty    4136
Name: count, dtype: int64


In [14]:
sex_pct = df['sex'].value_counts(normalize=True) * 100
sex_pct

,proportion
sex,
Male,66.92381
Female,33.07619


In [15]:
income_pct = df['income'].value_counts(normalize=True) * 100
income_pct

,proportion
income,
<=50K,75.907428
>50K,24.092572


## Task 8 — Cross-Check a Data Quality Assumption


In [16]:
edu_map = df.groupby('education')['education.num'].unique()
edu_map

,education.num
education,
10th,[6]
11th,[7]
12th,[8]
1st-4th,[2]
5th-6th,[3]
7th-8th,[4]
9th,[5]
Assoc-acdm,[12]
Assoc-voc,[11]


In [17]:
inconsistent = edu_map[edu_map.apply(len) > 1]
print('Education levels with inconsistent education.num mapping:', len(inconsistent))
inconsistent

Education levels with inconsistent education.num mapping: 0


,education.num
education,


## Task 9 — SQL-Based Data Extraction
Load the cleaned CSV into a local SQLite table, then extract data via a SQL query instead of reading the CSV directly.

In [23]:
conn = sqlite3.connect('adult_income.db')
df.to_sql('adult_income', conn, if_exists='replace', index=False)

print('Loaded into SQLite table adult_income')

Loaded into SQLite table adult_income


In [24]:
query = "SELECT * FROM adult_income WHERE age > 30;"
df_sql = pd.read_sql_query(query, conn)

print('Original cleaned DataFrame shape:', df.shape)
print('SQL query result shape (age > 30):', df_sql.shape)

# Sanity check: does the SQL result match the equivalent pandas filter?
pandas_filtered = df[df['age'] > 30]
print('Matches pandas-filtered shape:', df_sql.shape == pandas_filtered.shape)
# Extra check: does the FULL table (no filter) match the cleaned CSV exactly?
# This confirms the CSV -> SQLite load step itself lost or duplicated no rows,
# separately from the age > 30 filter query above.
df_full_from_sql = pd.read_sql_query("SELECT * FROM adult_income;", conn)

print('Cleaned CSV / DataFrame shape:', df.shape)
print('Full table pulled back from SQLite:', df_full_from_sql.shape)
print('Match:', df.shape == df_full_from_sql.shape)

Original cleaned DataFrame shape: (32537, 15)
SQL query result shape (age > 30): (21979, 15)
Matches pandas-filtered shape: True
Cleaned CSV / DataFrame shape: (32537, 15)
Full table pulled back from SQLite: (32537, 15)
Match: True


In [25]:
conn.close()

## Task 10 — Data Profiling Summary Report

Dataset overview: Cleaned dataset has 32,537 rows (32,561 originally, minus 24 duplicates) and 15 columns from the UCI Adult/Census Income dataset. Each row is one individual with demographic and employment attributes, plus a label showing if income is above or below 50K/year.

Data quality issues found:
- Hidden missing values: 4,262 cells stored as "?" instead of NaN, mainly in occupation (1,843), workclass (1,836), and native country (583)
- Duplicates: 24 exact duplicate rows removed; 1 row matching on all attributes except income was kept
- Inconsistent categories: no whitespace/casing issues found in education, marital status, or native country; education and education-num mappings were fully consistent

Key observations:
- Income label is imbalanced: about 76% <=50K vs 24% >50K
- Dataset skews male (67%) vs female (33%)
- Prof-specialty is the most common occupation
- Age (mean 38.6, median 37) and hours per week (mean 40.4, median 40) are both roughly symmetric

Cleaning decisions made:
- Replaced "?" placeholders with real NaN
- Filled workclass/occupation with "Unknown" instead of dropping or using the mode, to avoid biasing frequency counts
- Filled native country with its mode, since missingness there was low and one category dominates
- Dropped 24 exact duplicate rows; kept the 1 row differing only in income
- Stripped whitespace from key categorical columns as a precaution